# 01 - Data cleaning

**Project:** predicting the entry price on courier rate cards (Rate Simulator capstone)
**Notebook order:** 01 cleaning, 02 exploration, 03 modelling, 04 findings

This notebook takes the synthetic lane table produced by `src/generate_synthetic_data.py` and gets it ready for analysis. The table is simulated: it copies the layout of a courier rate-card export, but no real customer or price is in it. Each lane is one origin-to-destination price on one customer contract. The thing we want to predict later is `base_rate_inr`, the price of the first weight slab.

Cleaning decisions are written down as we go, so a reviewer can disagree with any of them.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

DATA_DIR = Path("../data/processed")
lanes = pd.read_csv(DATA_DIR / "rate_lanes.csv", parse_dates=["effective_date"])
lanes.shape

(1296, 19)

## 1. First look

In [2]:
lanes.head()

,customer_id,contract_id,lead_region,effective_date,product_code,product_type,charge_basis,freight_type,origin_type,dest_type,origin_zone,dest_zone,min_weight_kg,vol_divisor,n_slabs,first_slab_end_kg,add_weight_kg,add_rate_inr,base_rate_inr
0,CUST-001,CTR-016,WEST1,2025-09-01,I,1,Per Half KG,Z2ZI,ZONE,ZONE,IND,3,0.5,5000,40,0.5,0.5,1842.24,2097.24
1,CUST-001,CTR-015,SOUTH1,2025-06-01,D,2,Per Half KG,Z2Z,ZONE,ZONE,A,L,0.5,5000,2,0.5,0.5,26.21,28.16
2,CUST-001,CTR-001,SOUTH1,2026-02-01,D,2,Per Half KG,Z2Z,ZONE,ZONE,H,C,0.5,6000,3,0.5,0.5,16.73,19.81
3,CUST-001,CTR-016,WEST1,2025-09-01,D,2,Per Half KG,Z2Z,ZONE,ZONE,L,J,1.0,6000,3,0.5,0.5,26.55,35.07
4,CUST-001,CTR-022,SOUTH1,2025-09-01,D,2,Per Half KG,L2L,AREA,AREA,A,L,0.5,5000,2,0.5,0.5,17.61,24.94


In [3]:
lanes.dtypes

customer_id                     str
contract_id                     str
lead_region                     str
effective_date       datetime64[us]
product_code                    str
product_type                  int64
charge_basis                    str
freight_type                    str
origin_type                     str
dest_type                       str
origin_zone                     str
dest_zone                       str
min_weight_kg               float64
vol_divisor                   int64
n_slabs                       int64
first_slab_end_kg           float64
add_weight_kg               float64
add_rate_inr                float64
base_rate_inr               float64
dtype: object

One thing stands out already: codes such as `product_type` were read as numbers, but they are labels, so I will treat them as text. `effective_date` parsed cleanly, which is worth confirming before any month-based feature is built.

In [4]:
print("Missing values per column")
print(lanes.isna().sum()[lanes.isna().sum() > 0])
print()
print("Exact duplicate rows:", lanes.duplicated().sum())
print("Date range:", lanes["effective_date"].min().date(), "to", lanes["effective_date"].max().date())
print("Unparsed dates:", lanes["effective_date"].isna().sum())

Missing values per column
add_rate_inr    206
dtype: int64

Exact duplicate rows: 0
Date range: 2024-09-01 to 2026-05-01
Unparsed dates: 0


### Missing values

`add_rate_inr` is blank for some lanes. These are lanes whose price list has no per-extra-weight step, so a blank means "not applicable". I fill it with 0 and keep a flag so nothing is hidden. The column is **not** used as a model input anyway, because the add-on rate is part of the price we are trying to predict and would leak the answer.

In [5]:
lanes["has_add_rate"] = lanes["add_rate_inr"].notna()
lanes["add_rate_inr"] = lanes["add_rate_inr"].fillna(0)

# an open-ended first slab was stored as NaN; give it a large, obvious value
print("Open-ended first slabs:", lanes["first_slab_end_kg"].isna().sum())

Open-ended first slabs: 0


## 2. Types and labels

In [6]:
label_cols = ["product_code", "product_type", "charge_basis", "freight_type",
              "origin_type", "dest_type", "origin_zone", "dest_zone", "lead_region"]
for col in label_cols:
    lanes[col] = lanes[col].astype(str).str.strip().str.upper()

lanes[label_cols].nunique()

product_code     4
product_type     2
charge_basis     2
freight_type     4
origin_type      3
dest_type        3
origin_zone     13
dest_zone       22
lead_region      5
dtype: int64

Zone and region labels are now trimmed and upper-cased, so `a` and `A ` cannot become two categories.

The product letters stand for rate products in a courier's price list. I make no claim about what each letter means beyond what the data shows: product `I` is priced far above the others, which is how an international product would look, so I treat it as its own group.

## 3. Range checks

In [7]:
checks = {
    "base rate is positive": (lanes["base_rate_inr"] > 0).all(),
    "minimum weight is positive": (lanes["min_weight_kg"] > 0).all(),
    "volumetric divisor is 5000 or 6000": lanes["vol_divisor"].isin([5000, 6000]).all(),
    "first slab ends after it starts": (lanes["first_slab_end_kg"].dropna() > 0).all(),
}
for name, passed in checks.items():
    print(f"{'OK  ' if passed else 'FAIL'} {name}")

OK   base rate is positive
OK   minimum weight is positive
OK   volumetric divisor is 5000 or 6000
OK   first slab ends after it starts


## 4. Outliers

In [8]:
def count_outliers(log_values):
    q1, q3 = log_values.quantile([0.25, 0.75])
    spread = q3 - q1
    return int(((log_values < q1 - 1.5 * spread) | (log_values > q3 + 1.5 * spread)).sum())

log_rate = np.log(lanes["base_rate_inr"])
print("Outliers across all lanes (log scale):", count_outliers(log_rate))
print()
print("Outliers inside each product group:")
print(log_rate.groupby(lanes["product_code"]).apply(count_outliers).to_string())

Outliers across all lanes (log scale): 0

Outliers inside each product group:
product_code
A    0
D    6
I    6
V    0


Across the whole table nothing looks extreme, because the cheap and expensive groups are both large and the log scale pulls them together. Inside each product group a few lanes stand out. They are lanes priced higher or lower than the rest of their product, the sort of thing a pricing analyst would want flagged, so I keep every row. Dropping them would remove the lanes where a pricing analyst most needs help.

## 5. Features added during cleaning

In [9]:
lanes["same_zone"] = (lanes["origin_zone"] == lanes["dest_zone"]).astype(int)
lanes["effective_month"] = lanes["effective_date"].dt.month
lanes["months_since_2025"] = (
    (lanes["effective_date"].dt.year - 2025) * 12 + lanes["effective_date"].dt.month
)
lanes["log_base_rate"] = np.log(lanes["base_rate_inr"])
lanes[["same_zone", "effective_month", "months_since_2025", "log_base_rate"]].describe().round(2)

,same_zone,effective_month,months_since_2025,log_base_rate
count,1296.00,1296.0,1296.00,1296.00
mean,0.07,6.8,7.64,4.74
std,0.26,3.7,5.81,1.91
min,0.00,1.0,-3.00,2.50
25%,0.00,4.0,4.00,3.35
50%,0.00,8.0,9.00,3.65
75%,0.00,10.0,12.00,7.23
max,1.00,12.0,17.00,9.10


## 6. Save the clean table

In [10]:
clean_path = DATA_DIR / "rate_lanes_clean.csv"
lanes.to_csv(clean_path, index=False)
print(f"Saved {len(lanes):,} rows and {lanes.shape[1]} columns to {clean_path.name}")

Saved 1,296 rows and 24 columns to rate_lanes_clean.csv


### Summary of what changed

| Step | Action |
|---|---|
| Missing add-on rates | Filled with 0, flagged in `has_add_rate` |
| Labels | Trimmed, upper-cased, stored as text |
| Range checks | All passed |
| Outliers | None across the table; a few inside product groups, kept |
| New columns | `same_zone`, `effective_month`, `months_since_2025`, `log_base_rate` |

Next: `02_exploratory_analysis.ipynb`.